# M1-B2 — Classifieur de motifs d'insatisfaction

Une section par tâche du brief. Les cellules `TODO` sont à compléter.

In [20]:
import os
os.environ["KERAS_BACKEND"] = "torch"   # avant tout « import keras » : Keras utilisera PyTorch

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
CACHE = DATA / "cache"
CACHE.mkdir(exist_ok=True)
MODELS = ROOT / "models"

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

## Tâche 1 — Démarrer

Cette cellule télécharge le modèle d'embeddings au premier lancement (environ 470 Mo) 

In [13]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer(MODEL_NAME)
vecteurs = encoder.encode(["Le lit était très inconfortable.", "Le matelas était défoncé."])
vecteurs.shape   # (2, 384) : deux phrases, 384 nombres chacune

(2, 384)

## Tâche 2 — Explorer le corpus

In [ ]:
avis = pd.read_csv(DATA / "avis_motifs.csv")
recents = pd.read_csv(DATA / "avis_recents.csv")
avis.head()

In [14]:
avis["longueur_texte"] = avis["texte"].str.len()
avis_par_motif = avis.sort_values(["motif", "avis_id"])

pd.set_option("display.max_colwidth", 200)
echantillon = avis.sample(20, random_state=0).sort_values([ "avis_id","motif"])
display(echantillon[["avis_id", "texte","motif",  "note", "longueur_texte"]])

,avis_id,texte,motif,note,longueur_texte
45,46,Les portes claquent dans le couloir toute la nuit. On entendait la tuyauterie à chaque fois qu'un voisin tirait la chasse. On attendait mieux.,bruit,1,142
52,53,Arrivés tard le soir après un long trajet. Le check-in n'était possible qu'à 17 h au lieu de 15 h comme annoncé. La caution n'a toujours pas été débloquée trois semaines après. L'hôtel est situé a...,reservation,2,221
186,187,"Aucun commerce ni restaurant aux alentours. Chambre au-dessus de la discothèque, nuit blanche garantie. La fenêtre n'isole absolument pas du bruit du tramway. Dommage.",bruit,1,167
198,199,"Nous venons souvent, mais cette fois non. Impossible de dormir à cause du bruit de la rue.",bruit,1,90
204,205,"chambre au sous-sol, humide et sans fenetre. pas assez de rangements, et pas un seul cintre. on attendait mieux.",bruit,1,112
263,264,"Vacances en famille, une semaine. Tout est payant : la serviette de piscine, le transat, le coffre.",prix,2,99
303,304,la douche fuyait et inondait la salle de bain. on attendait mieux.,equipements,1,66
311,312,Quelle déception ! La chambre était spacieuse. La facture ne correspondait pas au prix de la réservation. Overbooké : on nous a relogés dans un autre hôtel.,reservation,2,156
328,329,Le veilleur de nuit dormait à la réception et ne répondait pas. On nous a répondu sèchement quand nous avons signalé un problème.,accueil,1,129
362,363,"Nous avions choisi cet hôtel pour fêter notre anniversaire de mariage. À ce tarif, on attend bien mieux. Une odeur d'égout dans la salle de bain toute la nuit. On a trouvé des miettes et un vieux ...",proprete,1,303


### Notre avis sur les étiquettes du LLM

| avis_id | Étiquette | Observation | Étiquette proposée|
|---|---|---|---|
| 444 | `accueil` | Manque de rangements et de cintres | `chambre` ou `equipements` |
| 205 | `bruit` | Chambre humide, sans fenêtre et avec peu de rangements ; aucun bruit mentionné | `chambre` |
| 988 | `restauration` | Critique du personnel, sans mention de nourriture | `accueil` |

## Tâche 3 — Une référence : TF-IDF + régression logistique

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score

train, test = train_test_split(avis, test_size=0.25, stratify=avis["motif"], random_state=42)

def f1_macro(y_true, y_pred):
    return round(f1_score(y_true, y_pred, average="macro"), 3)

tfidf = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000))
tfidf.fit(train["texte"], train["motif"])
print("TF-IDF — F1 macro test :", f1_macro(test["motif"], tfidf.predict(test["texte"])))


TF-IDF — F1 macro test : 0.911


c:\M1-B2-motifs-Hamza\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


## Tâche 4 — Des embeddings

Encode les avis **une seule fois**, puis sauvegarde les vecteurs (par exemple `np.save(CACHE / "avis.npy", ...)`)
pour ne pas les recalculer à chaque essai.

In [ ]:
# TODO : encoder avis et récents (une fois), sauvegarder, régression logistique sur les vecteurs

## Tâche 5 — Un petit réseau Keras

In [ ]:
#Note : les données préparées E_train et train proviennent des cellules précédentes

import keras
from sklearn.preprocessing import LabelEncoder


keras.utils.set_random_seed(42)
le = LabelEncoder().fit(avis["motif"])
y_train = le.transform(train["motif"])

reseau = keras.Sequential([
    keras.Input(shape=(E_train.shape[1],)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(len(le.classes_), activation="softmax"),
])
reseau.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])


historique = reseau.fit(
    E_train, y_train, validation_split=0.2, epochs=100, batch_size=32, verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
)
print("arrêt après", len(historique.history["loss"]), "époques")



## Tâche 6 — Comparer et analyser

In [ ]:
# TODO : F1 macro sur le test ET sur les avis récents, matrices de confusion, temps de prédiction, taille

### 10 erreurs examinées

| avis_id | étiquette | prédiction | erreur du modèle ou étiquette douteuse ? |
|---|---|---|---|
| | | | |

In [ ]:
# TODO : sauvegarder le modèle retenu dans MODELS